In [2]:
import networkx as nx
import pandas as pd
import numpy as np
from rdflib import Graph, URIRef
import pickle

## Process NP-KG and reference data for embeddings pipeline

In [4]:
refdf = pd.read_csv('../reference-data/reference_data_combined_nodrugclass_202402281258.tsv', sep='\t')
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 965 entries, 0 to 964
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                965 non-null    object
 1   np_id             965 non-null    object
 2   np                965 non-null    object
 3   drug_id           965 non-null    object
 4   drug              965 non-null    object
 5   interaction_type  964 non-null    object
 6   source_data       965 non-null    object
 7   adr               47 non-null     object
 8   adr_id            33 non-null     object
 9   source_id         965 non-null    int64 
dtypes: int64(1), object(9)
memory usage: 75.5+ KB


In [6]:
refdf['interaction_type'] = refdf['interaction_type'].str.lower()
refdf['interaction_type'].value_counts()

positive                                                                                              723
negative                                                                                               82
positive|positive                                                                                      80
positive|positive|positive                                                                             25
negative|negative                                                                                      12
negative|positive                                                                                      12
positive|negative                                                                                       7
positive|positive|positive|positive|positive|positive                                                   4
positive|negative|positive                                                                              3
positive|positive|positive|positive           

In [7]:
refdf.loc[refdf['interaction_type'] == 'positive|positive', 'interaction_type'] = 'positive'
refdf.loc[refdf['interaction_type'] == 'negative|negative', 'interaction_type'] = 'negative'
refdf.loc[refdf['interaction_type'] == 'positive|positive|positive', 'interaction_type'] = 'positive'
refdf.loc[refdf['interaction_type'] == 'negative|positive', 'interaction_type'] = 'both'
refdf.loc[refdf['interaction_type'] == 'positive|negative', 'interaction_type'] = 'both'
refdf.loc[refdf['interaction_type'] == 'positive|positive|positive|positive|positive|positive', 'interaction_type'] = 'positive'
refdf.loc[refdf['interaction_type'] == 'positive|negative|positive', 'interaction_type'] = 'both'
refdf.loc[refdf['interaction_type'] == 'positive|positive|negative', 'interaction_type'] = 'both'
refdf.loc[refdf['interaction_type'] == 'negative|positive|positive', 'interaction_type'] = 'both'
refdf.loc[refdf['interaction_type'] == 'positive|positive|positive|positive', 'interaction_type'] = 'positive'
refdf.loc[refdf['interaction_type'] == 'positive|positive|positive|negative', 'interaction_type'] = 'both'
refdf.loc[refdf['interaction_type'] == 'positive|negative|positive|positive', 'interaction_type'] = 'both'
refdf.loc[refdf['interaction_type'] == 'positive|positive|positive|positive|positive|positive|negative|positive', 'interaction_type'] = 'both'
refdf.loc[refdf['interaction_type'] == 'positive|negative|negative', 'interaction_type'] = 'both'
refdf.loc[refdf['interaction_type'] == 'positive|positive|positive|positive|positive|positive|positive|positive|positive|positive|positive', 'interaction_type'] = 'positive'
refdf.loc[refdf['interaction_type'] == 'negative|negative|negative', 'interaction_type'] = 'negative'
refdf.loc[refdf['interaction_type'] == 'positive|positive|both|negative', 'interaction_type'] = 'both'
refdf.loc[refdf['interaction_type'] == 'positive|positive|positive|positive|positive', 'interaction_type'] = 'positive'

In [8]:
refdf['interaction_type'].value_counts()

positive    837
negative     96
both         31
Name: interaction_type, dtype: int64

In [9]:
##import graph
KG_PATH = '/home/sanya/np-kg/resources/knowledge_graphs/'
NodeLabelsFile = KG_PATH + 'nodeLabels_v2.0.0.pickle'
#KG_NAME_MERGED = 'NP-KG_v2.0.0.gpickle'
KG_NAME_MERGED_WEIGHTED = 'NP-KG_BETWEENNESS-WEIGHTED-PK_v2.0.0.gpickle'
DIR_OUT = './output/'

In [10]:
with open(KG_PATH+KG_NAME_MERGED_WEIGHTED, 'rb') as f:
    npkg = pickle.load(f)

In [11]:
with open(NodeLabelsFile, 'rb') as filep:
    nodeLabels = pickle.load(filep)

In [14]:
def get_graph_stats(kg):
    nodes = nx.number_of_nodes(kg)
    edges = nx.number_of_edges(kg)
    self_loops = nx.number_of_selfloops(kg)

    print('There are {} nodes, {} edges, and {} self-loop(s)'.format(nodes, edges, self_loops))
    # get degree information
    avg_degree = float(edges) / nodes
    print('The Average Degree is {}'.format(avg_degree))
    
    print('Nodes with highest degree:')
    n_deg = sorted([(str(x[0]), x[1]) for x in  kg.degree], key=lambda x: x[1], reverse=1)[:6]

    for x in n_deg:
        print('Label: {}'.format(nodeLabels[x[0]]))
        print('{} (degree={})'.format(x[0], x[1]))
    # get network density
    density = nx.density(kg)

    print('The density of the graph is: {}'.format(density))

In [15]:
get_graph_stats(npkg)

There are 1090172 nodes, 7920893 edges, and 606 self-loop(s)
The Average Degree is 7.265727793412416
Nodes with highest degree:
Label: SNV
http://purl.obolibrary.org/obo/SO_0001483 (degree=214261)
Label: protein_coding_gene
http://purl.obolibrary.org/obo/SO_0001217 (degree=189453)
Label: transcript
http://purl.obolibrary.org/obo/SO_0000673 (degree=178302)
Label: Homo sapiens
http://purl.obolibrary.org/obo/NCBITaxon_9606 (degree=91535)
Label: protein
http://purl.obolibrary.org/obo/PR_000000001 (degree=56340)
Label: Mus musculus
http://purl.obolibrary.org/obo/NCBITaxon_10090 (degree=53645)
The density of the graph is: 6.664759742657267e-06


In [12]:
refdf.head()

,id,np_id,np,drug_id,drug,interaction_type,source_data,adr,adr_id,source_id
0,stockley_1,http://napdi.org/napdi_srs_imports:panax_ginseng,Panax Ginseng,http://purl.obolibrary.org/obo/CHEBI_16664,Albendazole,positive,Stockleys,NaN,NaN,1
1,stockley_2,http://napdi.org/napdi_srs_imports:allium_sativum,Garlic,http://purl.obolibrary.org/obo/CHEBI_30879,Alcohol,positive,Stockleys,NaN,NaN,2
2,stockley_3,http://napdi.org/napdi_srs_imports:panax_ginseng,Panax Ginseng,http://purl.obolibrary.org/obo/CHEBI_30879,Alcohol,positive,Stockleys,NaN,NaN,3
3,stockley_4|natmed_187,http://napdi.org/napdi_srs_imports:valeriana_o...,Valerian|Valerian,http://purl.obolibrary.org/obo/CHEBI_30879,Alcohol|ALCOHOL (Ethanol),positive,Stockleys|NatMedPro,increased sedative effect,NaN,4
4,stockley_5|napdi_245,http://napdi.org/napdi_srs_imports:camellia_si...,Green Tea|green tea leaf extract,http://purl.obolibrary.org/obo/CHEBI_2611,Alprazolam|alprazolam,negative,Stockleys|NaPDI,NaN,NaN,5


In [13]:
refdf['npkg_edge'] = np.nan
refdf['npkg_edge_label'] = np.nan

In [16]:
for i in range(len(refdf.index)):
    edge_list = []
    edge_label_list = []
    np_id = refdf.loc[i, 'np_id']
    drug_id = refdf.loc[i, 'drug_id']
    source = URIRef(np_id)
    target = URIRef(drug_id)
    if source in npkg and target in npkg:
        if npkg.has_edge(source, target):
            edge_data = npkg.get_edge_data(source, target)
            for k in edge_data:
                edge_list.append(k)
                edge_label_list.append(nodeLabels[str(k)])
    if edge_list:
        edge_list = '|'.join(edge_list)
        edge_label_list = '|'.join(edge_label_list)
        refdf.at[i, 'npkg_edge'] = edge_list
        refdf.at[i, 'npkg_edge_label'] = edge_label_list
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 965 entries, 0 to 964
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                965 non-null    object
 1   np_id             965 non-null    object
 2   np                965 non-null    object
 3   drug_id           965 non-null    object
 4   drug              965 non-null    object
 5   interaction_type  964 non-null    object
 6   source_data       965 non-null    object
 7   adr               47 non-null     object
 8   adr_id            33 non-null     object
 9   source_id         965 non-null    int64 
 10  npkg_edge         34 non-null     object
 11  npkg_edge_label   34 non-null     object
dtypes: int64(1), object(11)
memory usage: 90.6+ KB


In [17]:
refdf.loc[refdf['npkg_edge'].notnull()]

,id,np_id,np,drug_id,drug,interaction_type,source_data,adr,adr_id,source_id,npkg_edge,npkg_edge_label
13,stockley_45|napdi_229,http://napdi.org/napdi_srs_imports:camellia_si...,Green Tea|green tea leaf,http://purl.obolibrary.org/obo/CHEBI_27732,Caffeine|caffeine,negative,Stockleys|NaPDI,NaN,NaN,14,http://purl.obolibrary.org/obo/RO_0002180|http...,has component|directly positively regulates qu...
36,stockley_69|napdi_69|napdi_70|napdi_71,http://purl.obolibrary.org/obo/CHEBI_16118,Berberine|berberine|berberine|berberine,http://purl.obolibrary.org/obo/CHEBI_4031,Ciclosporin|cyclosporin a|cyclosporin a|cyclos...,both,Stockleys|NaPDI|NaPDI|NaPDI,NaN,NaN,37,http://purl.obolibrary.org/obo/RO_0002213|http...,positively regulates|directly negatively regul...
125,stockley_174|natmed_16|fideo_310,http://napdi.org/napdi_srs_imports:camellia_si...,Green Tea|Green tea| tea food product,http://purl.obolibrary.org/obo/CHEBI_10033,Warfarin and related drugs|WARFARIN (Coumadin)...,both,Stockleys|NatMedPro|FIDEO,bleeding,http://purl.obolibrary.org/obo/HP_0001892,126,http://purl.obolibrary.org/obo/RO_0002449,directly negatively regulates activity of
126,stockley_175|natmed_232|fideo_452|fideo_453,http://napdi.org/napdi_srs_imports:panax_ginseng,Panax Ginseng|Panax ginseng| ginseng plant foo...,http://purl.obolibrary.org/obo/CHEBI_10033,Warfarin and related drugs|WARFARIN (Coumadin)...,both,Stockleys|NatMedPro|FIDEO|FIDEO,NaN,NaN,127,http://purl.obolibrary.org/obo/RO_0002448,directly regulates activity of
128,napdi_0|napdi_259,http://purl.obolibrary.org/obo/CHEBI_69919,(-)-beta-hydrastine|hydrastine,http://purl.obolibrary.org/obo/CHEBI_47381,diclofenac|diclofenac,positive,NaPDI|NaPDI,NaN,NaN,129,http://purl.obolibrary.org/obo/RO_0002436,molecularly interacts with
130,napdi_2|napdi_261|napdi_269,http://purl.obolibrary.org/obo/CHEBI_69919,(-)-beta-hydrastine|hydrastine|hydrastine,http://purl.obolibrary.org/obo/CHEBI_17347,testosterone|testosterone|testosterone,positive,NaPDI|NaPDI|NaPDI,NaN,NaN,131,http://purl.obolibrary.org/obo/RO_0002436,molecularly interacts with
132,napdi_4|napdi_268|napdi_271,http://purl.obolibrary.org/obo/CHEBI_69919,(-)-beta-hydrastine|hydrastine|hydrastine,http://purl.obolibrary.org/obo/CHEBI_6931,midazolam|midazolam|midazolam,positive,NaPDI|NaPDI|NaPDI,NaN,NaN,133,http://purl.obolibrary.org/obo/RO_0002449|http...,directly negatively regulates activity of|prot...
145,napdi_26|napdi_137,http://purl.obolibrary.org/obo/CHEBI_4806,(-)-epigallocatechin gallate|epigallocatechin ...,http://purl.obolibrary.org/obo/CHEBI_6931,midazolam|midazolam,positive,NaPDI|NaPDI,NaN,NaN,146,http://purl.obolibrary.org/obo/RO_0002436|http...,molecularly interacts with|directly negatively...
146,napdi_27,http://purl.obolibrary.org/obo/CHEBI_4806,(-)-epigallocatechin gallate,http://purl.obolibrary.org/obo/CHEBI_7444,nadolol,positive,NaPDI,NaN,NaN,147,http://purl.obolibrary.org/obo/RO_0002449|http...,directly negatively regulates activity of|dire...
148,napdi_30|napdi_31,http://purl.obolibrary.org/obo/CHEBI_4806,(-)-epigallocatechin gallate|(-)-epigallocatec...,http://purl.obolibrary.org/obo/CHEBI_63836,bromosulfophthalein|bromosulfophthalein,both,NaPDI|NaPDI,NaN,NaN,149,http://purl.obolibrary.org/obo/RO_0002449,directly negatively regulates activity of


In [18]:
refdf.to_csv('reference_data_combined_interactions_nodrugclass_dedup_20240327_with_npkg_edges.tsv', sep='\t', index=False)

In [19]:
refdf_nokg = refdf.loc[refdf['npkg_edge'].isnull()]
refdf_nokg.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 931 entries, 0 to 964
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                931 non-null    object
 1   np_id             931 non-null    object
 2   np                931 non-null    object
 3   drug_id           931 non-null    object
 4   drug              931 non-null    object
 5   interaction_type  930 non-null    object
 6   source_data       931 non-null    object
 7   adr               46 non-null     object
 8   adr_id            32 non-null     object
 9   source_id         931 non-null    int64 
 10  npkg_edge         0 non-null      object
 11  npkg_edge_label   0 non-null      object
dtypes: int64(1), object(11)
memory usage: 94.6+ KB


In [20]:
refdf_nokg.to_csv('reference_data_combined_interactions_nodrugclass_dedup_20240327_without_npkg_edges.tsv', sep='\t', index=False)